In [ ]:
import pandas as pd
import re

# 1. Load your dataset
# (Ensure the path matches your Kaggle environment)
df = pd.read_csv('/kaggle/input/notebooks/shivanguniyal/cleaning-hindi-corpus-with-boilerplates/Hindi_Corpus_Cleaned_NoTruncation.csv')

# 2. Define the metadata cleaning function
def clean_metadata_prefix(text):
    if not isinstance(text, str):
        return text
        
    # Layer 1: Catches "Edited By [Name],Updated: [Date] [Time] AM/PM"
    # The .*? ensures it stops at the first AM/PM, preventing it from eating the whole article
    text = re.sub(r'^Edited By\s+.*?(?:AM|PM)\s*', '', text, flags=re.IGNORECASE)
    
    # Layer 2: Catches texts that start directly with "Updated: [Date] [Time] AM/PM"
    text = re.sub(r'^Updated:\s*.*?(?:AM|PM)\s*', '', text, flags=re.IGNORECASE)
    
    # Layer 3: Catch-all for stray "Edited By [Name]," at the start without any timestamp
    text = re.sub(r'^Edited By\s+[^,]+,\s*', '', text, flags=re.IGNORECASE)
    
    # Layer 4: Remove stray "AM " or "PM " left at the start of the text by the scraper
    # (e.g., "AM भगवान विष्णु" -> "भगवान विष्णु" | "PM प्रधानमंत्री" -> "प्रधानमंत्री")
    text = re.sub(r'^(AM|PM)\s+', '', text, flags=re.IGNORECASE)
    
    # Layer 5: Remove stray JSON metadata blocks at the start (e.g., {"_id":"...","slug":"..."})
    text = re.sub(r'^\{[^}]*\}\s*', '', text)
    
    # Layer 6: Remove "Hindi News" breadcrumbs if they appear at the very start
    text = re.sub(r'^-\s*Hindi News\s*-[^\n]*?(?=[ा-ॿ])', '', text)
    
    return text.strip()

# 3. Apply the cleaner to the 'clean_text' column
print("🧹 Removing 'Edited By / Updated' metadata prefixes and scraper artifacts...")
df['clean_text'] = df['clean_text'].apply(clean_metadata_prefix)

# 4. Save the pristine dataset
df.to_csv('/kaggle/working/FINAL_Hindi_Pristine_Corpus.csv', index=False)
print("✅ Metadata prefixes and artifacts successfully removed!")